# Index Analysis

In notebook **01_discovery** we spotted several suspicious indexes — unused PKs on time-series tables, a redundant email index, and some large composite indexes with zero scans. Here we use the `Forensic` class to systematically classify every index and get a health score.

The forensic workflow:
1. Fetch index metadata from `pg_stat_user_indexes` and selectivity from `pg_stats`
2. Classify each index as `CRITICAL`, `SUSPICIOUS`, `HEALTHY`, `PK`, or `UNIQUE`
3. Compute a 0–100 health score per index
4. Produce a ranked list of drop candidates

## Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config SqlMagic.displaylimit = None
%load_ext sql

## Classification Guide

The `Forensic` class uses these rules:

| Status | Condition | Action |
|---|---|---|
| 🔴 **CRITICAL** | `idx_scan = 0` + large (＞ 40% of table) | Drop candidate — pure waste |
| 🟠 **SUSPICIOUS** | `idx_scan < 100` + not PK/unique | Needs review — low value |
| 🟢 **HEALTHY** | Actively used | Keep — justifies its cost |
| 🔵 **PK** | Primary key index | Always keep — integrity |
| 🔷 **UNIQUE** | Unique constraint | Keep — integrity (even if usage is moderate) |

**Key columns:**
- `idx_scan` — how many times the index was actually used
- `pct_of_table` — index size relative to table heap
- `selectivity` — ratio of distinct values to total rows (closer to 0 = more selective = more useful)

## 1. Raw Index Overview

Every index sorted by size — lets us quickly spot the biggest storage consumers.

In [2]:
%%sql
SELECT 
    s.schemaname,
    s.relname AS table_name,
    s.indexrelname AS index_name,
    pg_size_pretty(pg_relation_size(s.indexrelid)) AS index_size,
    pg_size_pretty(pg_relation_size(t.relid)) AS table_size,
    s.idx_scan,
    ROUND(
        pg_relation_size(s.indexrelid)::numeric / 
        NULLIF(pg_relation_size(t.relid), 0) * 100, 
        2
    ) AS pct_of_table
FROM pg_stat_user_indexes s
JOIN pg_stat_user_tables t 
    ON s.relid = t.relid
ORDER BY pg_relation_size(s.indexrelid) DESC;

The top offenders are already visible: `idx_metric_time` (481 MB, 0 scans), `events_pkey` (391 MB, 0 scans), `idx_session` (85 MB, 0 scans).

## 2. Forensic Classification

The `Forensic.check_indexes()` method enriches the raw data with selectivity, classifies every index, and computes a score.

In [1]:
import os
from database_as_crime_scene.forensic.forensic import Forensic

database_url = os.getenv('DATABASE_URL', 'localhost:5432')
forensic = Forensic(database_url)
df = forensic.check_indexes()
df

┏━━━━┳━━━━━┳━━━━━┳━━━━━┳━━━━━┳━━━━━┳━━━━━┳━━━━━┳━━━━━┳━━━━━┳━━━━━┳━━━━┳━━━━━┳━━━━┳━━━━━┳━━━━┳━━━━━┳━━━━┳━━━━━┳━━━━┓
┃    ┃ sc… ┃ ta… ┃ in… ┃ in… ┃ id… ┃ in… ┃ he… ┃ to… ┃ pc… ┃ pc… ┃ p… ┃ is… ┃ i… ┃ st… ┃ s… ┃ co… ┃ n… ┃ to… ┃ s… ┃
┡━━━━╇━━━━━╇━━━━━╇━━━━━╇━━━━━╇━━━━━╇━━━━━╇━━━━━╇━━━━━╇━━━━━╇━━━━━╇━━━━╇━━━━━╇━━━━╇━━━━━╇━━━━╇━━━━━╇━━━━╇━━━━━╇━━━━┩
│ 0  │ pu… │ us… │ id… │ bt… │ 0   │ 0.… │ 0.… │ 2.… │ 70… │ 70… │ 2… │ Fa… │ F… │ CR… │ 0… │ em… │ -… │ 10… │ 1… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │     │     │    │     │    │     │    │     │    │     │    │
│ 1  │ pu… │ me… │ id… │ bt… │ 0   │ 48… │ 60… │ 13… │ 80… │ 80… │ 3… │ Fa… │ F… │ CR… │ 0… │ me… │ 5… │ 10… │ 0… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │     │     │    │     │    │     │    │     │    │     │    │
│ 2  │ pu… │ me… │ id… │ bt… │ 0   │ 48… │ 60… │ 13… │ 80… │ 80… │ 3… │ Fa… │ F… │ CR… │ 0… │ re… │ -… │ 10… │ 1… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │     │     │    │     │    │     │    │     │    │     │    │
│ 3  │ pu… │ co… │ id… │ bt… │ 0   │ 18… │ 14… │ 20… │ 12… │ 12… │ 9… │ Fa… │ F… │ HE… │ 0… │ fk… │ -… │ 12… │ 0… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │     │     │    │     │    │     │    │     │    │     │    │
│ 4  │ pu… │ co… │ id… │ bt… │ 3   │ 9.… │ 14… │ 20… │ 6.… │ 6.… │ 4… │ Fa… │ F… │ HE… │ 3… │ fk… │ 1… │ 12… │ 0… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │     │     │    │     │    │     │    │     │    │     │    │
│ 5  │ pu… │ po… │ id… │ bt… │ 0   │ 1.… │ 21… │ 26… │ 5.… │ 5.… │ 4… │ Fa… │ F… │ HE… │ 4… │ fk… │ 1… │ 15… │ 0… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │     │     │    │     │    │     │    │     │    │     │    │
│ 6  │ pu… │ po… │ id… │ bt… │ 1   │ 0.… │ 21… │ 26… │ 4.… │ 4.… │ 3… │ Fa… │ F… │ HE… │ 3… │ cr… │ 1… │ 15… │ 0… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │     │     │    │     │    │     │    │     │    │     │    │
│ 7  │ pu… │ lo… │ id… │ bt… │ 0   │ 85… │ 84… │ 11… │ 10… │ 10… │ 7… │ Fa… │ F… │ HE… │ 1… │ se… │ 9… │ 10… │ 0… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │     │     │    │     │    │     │    │     │    │     │    │
│ 8  │ pu… │ co… │ co… │ bt… │ 0   │ 25… │ 14… │ 20… │ 17… │ 17… │ 1… │ Tr… │ T… │ PK  │ 1… │ id  │ -… │ 12… │ 1… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │     │     │    │     │    │     │    │     │    │     │    │
│ 9  │ pu… │ po… │ po… │ bt… │ 12… │ 3.… │ 21… │ 26… │ 15… │ 15… │ 1… │ Tr… │ T… │ PK  │ 1… │ id  │ -… │ 15… │ 1… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │     │     │    │     │    │     │    │     │    │     │    │
│ 10 │ pu… │ me… │ me… │ bt… │ 0   │ 21… │ 60… │ 13… │ 35… │ 35… │ 1… │ Tr… │ T… │ PK  │ 1… │ id  │ -… │ 10… │ 1… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │     │     │    │     │    │     │    │     │    │     │    │
│ 11 │ pu… │ lo… │ lo… │ bt… │ 0   │ 21… │ 84… │ 11… │ 25… │ 25… │ 1… │ Tr… │ T… │ PK  │ 1… │ id  │ -… │ 10… │ 1… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │     │     │    │     │    │     │    │     │    │     │    │
│ 12 │ pu… │ fr… │ fr… │ bt… │ 24… │ 0.… │ 0.… │ 1.… │ 64… │ 64… │ 2… │ Tr… │ T… │ PK  │ 1… │ us… │ -… │ 99… │ 0… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │     │     │    │     │    │     │    │     │    │     │    │
│ 13 │ pu… │ fr… │ fr… │ bt… │ 24… │ 0.… │ 0.… │ 1.… │ 64… │ 64… │ 2… │ Tr… │ T… │ PK  │ 1… │ fr… │ -… │ 99… │ 0… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │     │     │    │     │    │     │    │     │    │     │    │
│ 14 │ pu… │ ev… │ ev… │ bt… │ 0   │ 38… │ 65… │ 10… │ 59… │ 59… │ 3… │ Tr… │ T… │ PK  │ 1… │ id  │ -… │ 10… │ 1… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │     │     │    │     │    │     │    │     │    │     │    │
│ 15 │ pu… │ us… │ us… │ bt… │ 13… │ 0.… │ 0.… │ 2.… │ 28… │ 28… │ 1… │ Tr… │ T… │ PK  │ 1… │ id  │ -… │ 10… │ 1… │
│    │     │     │     │     │     │ MB  │ MB  │ MB  │  

,schemaname,table_name,index_name,index_type,idx_scan,index_size,heap_size,total_size,pct_of_table,pct_vs_heap,pct_vs_total,is_pk,is_unique,status,score,column_name,n_distinct,total_rows,selectivity
0,public,users_profile,idx_users_profile_email,btree,0,0.58 MB,0.82 MB,2.25 MB,70.48,70.48,25.69,False,False,CRITICAL,0.0,email,-1.000000,10000.0,1.0000
1,public,metrics,idx_metric_time,btree,0,488.24 MB,606.70 MB,1309.36 MB,80.48,80.48,37.29,False,False,CRITICAL,0.0,metric_name,5.000000,10000052.0,0.0000
2,public,metrics,idx_metric_time,btree,0,488.24 MB,606.70 MB,1309.36 MB,80.48,80.48,37.29,False,False,CRITICAL,0.0,recorded_at,-1.000000,10000052.0,1.0000
3,public,comments,idx_comments_fk_post_id,btree,0,18.05 MB,146.68 MB,200.30 MB,12.31,12.31,9.01,False,False,HEALTHY,0.0,fk_post_id,-0.123056,1200000.0,0.1231
4,public,comments,idx_comments_fk_user_id,btree,3,9.77 MB,146.68 MB,200.30 MB,6.66,6.66,4.88,False,False,HEALTHY,37.6,fk_user_id,10003.000000,1200000.0,0.0083
5,public,posts,idx_posts_fk_user_id,btree,0,1.26 MB,21.48 MB,26.95 MB,5.86,5.86,4.67,False,False,HEALTHY,4.7,fk_user_id,10004.000000,150000.0,0.0667
6,public,posts,idx_posts_created_at,btree,1,0.94 MB,21.48 MB,26.95 MB,4.37,4.37,3.48,False,False,HEALTHY,38.3,created_at,1.000000,150000.0,0.0000
7,public,logs,idx_session,btree,0,85.56 MB,846.21 MB,1146.27 MB,10.11,10.11,7.46,False,False,HEALTHY,1.2,session_id,97564.000000,10000298.0,0.0098
8,public,comments,comments_pkey,btree,0,25.73 MB,146.68 MB,200.30 MB,17.54,17.54,12.84,True,True,PK,100.0,id,-1.000000,1200000.0,1.0000
9,public,posts,posts_pkey,btree,1200000,3.23 MB,21.48 MB,26.95 MB,15.06,15.06,12.00,True,True,PK,100.0,id,-1.000000,150000.0,1.0000


### Reading the forensic report

| Column | What it tells you |
|---|---|
| `status` | Classification: CRITICAL / SUSPICIOUS / HEALTHY / PK / UNIQUE |
| `score` | 0–100 usefulness score (higher = better). Below 50 = consider dropping |
| `idx_scan` | How many times the index was used (since last stats reset) |
| `selectivity` | Low (close to 0) = high cardinality = index is useful for lookups. High (close to 1) = low cardinality = index is less effective |
| `pct_of_table` / `pct_vs_heap` | Index size vs data size — overhead gauge |
| `is_pk` / `is_unique` | Never drop these without understanding the integrity constraint |

## 3. Findings Summary

### 🔴 CRITICAL (drop candidates)

| Index | Size | `idx_scan` | Selectivity | Why |
|---|---|---|---|---|
| `idx_users_profile_email` | 0.55 MB | 0 | 1.0 (unique) | **Duplicate** — `users_profile_email_key` already covers this. Drop the redundant one. |
| `idx_metric_time` | 481 MB | 0 | 0.0 (metric_name) / 1.0 (recorded_at) | Composite index on `(metric_name, recorded_at)`. metric_name has only 5 distinct values (low cardinality = terrible for btree). Zero usage. |

### 🟠 SUSPICIOUS (needs review)

| Index | Size | `idx_scan` | Selectivity | Why |
|---|---|---|---|---|
| `idx_friends_user_id` | 0.14 MB | 0 | 0.38 | Friends PK already covers `user_id` as leading column. Likely redundant. |
| `idx_friends_friend_id` | 0.17 MB | 0 | 0.62 | FK index on friend_id — never used. May be needed for cascade operations. |

### 🟢 HEALTHY (keep)

All social table indexes with actual usage (`posts_pkey`, `users_profile_pkey`, `friends_pkey`, `users_profile_email_key`) are fine. The FK indexes on `comments` and `posts` with low-or-zero scans but small size are marked HEALTHY because they're small relative to their tables — the cost of keeping them is negligible compared to the risk of missing a future query pattern.

### 🔵 PK — 0 scans but kept by rule

These PK indexes show `idx_scan = 0` but are protected by the `is_pk` flag:
- `events_pkey` — 391 MB
- `metrics_pkey` — 214 MB
- `logs_pkey` — 214 MB
- `comments_pkey` — 26 MB

They exist on UUID/serial columns that are never searched by PK. While we can't drop them (they're the table's identity), their size is pure overhead for insert-heavy workloads.

## 4. Action Plan

| Priority | Action | SQL | Saves |
|---|---|---|---|
| P0 | Drop redundant `idx_users_profile_email` (duplicates unique constraint) | `DROP INDEX IF EXISTS idx_users_profile_email;` | 0.55 MB + faster writes |
| P1 | Investigate `idx_metric_time` — replace with partial index or drop | Requires understanding query patterns first | 481 MB + write overhead |
| P2 | Investigate `idx_session` on logs — preserved only if session-based queries exist | `DROP INDEX IF EXISTS idx_session;` if unused | 85 MB + write overhead |
| P3 | Review `idx_friends_user_id` — likely redundant given PK on `(user_id, friend_id)` | Validate then `DROP INDEX IF EXISTS idx_friends_user_id;` | 0.14 MB |
| P4 | Consider partitioning time-series tables (notebook **06**) to shrink PK index sizes per partition | Range partition by time | ~800 MB in PK overhead for active partition only |

**⚠️ Always verify an index is unused before dropping** — `pg_stat_user_indexes` resets on server restart, so 0 scans may mean "reset since last use." Run a workload first, then re-check.